# Clase 2 — Primer contacto con un modelo de IA de Hugging Face

**Materia:** Procesamiento del Lenguaje Natural  
**Tema:** Uso inicial de un modelo de lenguaje desde Google Colab  
**Objetivo:** ejecutar una primera consulta a un modelo de IA y analizar sus respuestas.

En esta clase todavía **no trabajamos con RAG ni con documentos propios**.  
Primero vamos a probar cómo responde un modelo de lenguaje general.


## 1. Activar GPU en Colab

Antes de ejecutar el notebook:

1. Ir a **Entorno de ejecución**.
2. Elegir **Cambiar tipo de entorno de ejecución**.
3. Seleccionar **GPU T4**.
4. Guardar los cambios.

La GPU permite ejecutar modelos más grandes y obtener respuestas en menos tiempo.


In [ ]:
!pip -q install -U \
    "transformers>=4.51,<5" \
    accelerate \
    bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Importar librerías y verificar GPU

En esta celda verificamos si Colab tiene disponible una GPU.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No hay GPU disponible. Activá GPU T4 en Colab antes de continuar.")

GPU disponible: True
GPU: Tesla T4


## 3. Elegir el modelo

Vamos a usar un modelo de Hugging Face.

Importante:

- No estamos entrenando un modelo.
- Estamos usando un modelo ya entrenado.
- En esta clase solamente probamos cómo responde.


In [ ]:
MODELO_ID = "Qwen/Qwen3-4B-Instruct-2507"

print("Modelo seleccionado:", MODELO_ID)

Modelo seleccionado: Qwen/Qwen3-4B-Instruct-2507


## 4. Configurar carga en 4 bits

La cuantización permite cargar el modelo usando menos memoria.

Esto ayuda a que el modelo pueda ejecutarse en una GPU gratuita de Colab.


In [ ]:
config_cuantizacion = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Configuración de cuantización lista.")

Configuración de cuantización lista.


## 5. Cargar tokenizador y modelo

El **tokenizador** transforma el texto en tokens.

El **modelo** recibe esos tokens y genera una respuesta.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)

modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    quantization_config=config_cuantizacion,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

modelo.eval()

print("Modelo cargado correctamente.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

... (5 outputs left out)

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Modelo cargado correctamente.


## 6. Crear una función para preguntar al modelo

Esta función recibe una pregunta y devuelve una respuesta generada por el modelo.

Usamos un mensaje de sistema simple:

> Sos un asistente educativo de PLN. Respondé en español, claro y breve.


In [ ]:
def responder_con_ia(pregunta):
    mensajes = [
        {
            "role": "system",
            "content": "Sos un asistente educativo de PLN. Respondé en español, claro y breve."
        },
        {
            "role": "user",
            "content": pregunta
        }
    ]

    texto_chat = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        texto_chat,
        return_tensors="pt"
    ).to(modelo.device)

    with torch.inference_mode():
        salida = modelo.generate(
            **entradas,
            max_new_tokens=180,
            do_sample=True,
            temperature=0.3,
            top_p=0.85,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )

    tokens_nuevos = salida[0][entradas["input_ids"].shape[-1]:]

    respuesta = tokenizer.decode(
        tokens_nuevos,
        skip_special_tokens=True
    )

    return respuesta.strip()

## 7. Primera prueba

Probamos el modelo con una pregunta conceptual sobre la materia.


In [ ]:
pregunta = "¿Qué es el procesamiento del lenguaje natural?"
respuesta = responder_con_ia(pregunta)

print(respuesta)

El procesamiento del lenguaje natural (PLN) es una rama de la inteligencia artificial que permite a las máquinas entender, interpretar, generar y responder en lenguaje humano.


## 8. Segunda prueba

Ahora probamos una pregunta relacionada con chatbots.


In [ ]:
pregunta = "Explicá la diferencia entre un chatbot por reglas y un chatbot con IA."
respuesta = responder_con_ia(pregunta)

print(respuesta)

Claro:

- **Chatbot por reglas**: Responde a preguntas según una lista predefinida de situaciones y respuestas. Es simple y fijo, no entiende el contexto ni aprende. Por ejemplo: si dice "precio", responde "50 dólares". Si pregunta algo fuera del listado, no sabe qué hacer.

- **Chatbot con IA (Inteligencia Artificial)**: Usa modelos de aprendizaje automático para entender el lenguaje natural, aprender de conversaciones reales y responder de forma más flexible y natural. Puede entender tono, contexto y hacer predicciones basadas en patrones.

En resumen:  
Reglas = respuestas preprogramadas.  
IA = aprende y entiende como un humano.


# Actividad de la clase

La actividad **no está resuelta**: lo que sigue es una plantilla.

Cada estudiante debe adaptar las preguntas a su propio proyecto de chatbot.

La idea es probar el modelo con tres tipos de preguntas:

1. Una pregunta conceptual sobre PLN.
2. Una pregunta aplicada al futuro chatbot.
3. Una pregunta problemática o de control.


## ¿Qué es una pregunta problemática?

Una pregunta problemática es una pregunta donde el modelo podría **inventar información** porque todavía no le dimos documentos ni fuentes oficiales.

Ejemplos:

- ¿Cuáles son las fechas oficiales de inscripción 2026?
- ¿Cuánto cuesta la cuota mensual?
- ¿Cuál es el horario exacto de atención de una institución?
- ¿Qué medicamento debería tomar si tengo fiebre?
- ¿Cuál es el precio exacto de una entrada este mes?

En esta clase todavía no esperamos que el modelo resuelva todo correctamente.  
Queremos observar si responde con seguridad aunque no tenga información confiable.


## 9. Actividad guiada: completar con preguntas propias

Reemplazá los textos entre corchetes por preguntas relacionadas con tu proyecto.


In [ ]:
mis_preguntas = [
    # 1. Pregunta conceptual sobre PLN
    "¿Qué es la tokenización?",

    # 2. Pregunta aplicada al proyecto del alumno
    "¿Cómo podría ayudar un chatbot a estudiantes de una biblioteca?",

    # 3. Pregunta problemática: el modelo no debería inventar este dato
    "¿Cuáles son las fechas oficiales de inscripción 2026 de una institución que no fue cargada como fuente?"
]

for numero, pregunta in enumerate(mis_preguntas, start=1):
    print("=" * 80)
    print(f"PREGUNTA {numero}: {pregunta}")
    print("-" * 80)
    respuesta = responder_con_ia(pregunta)
    print(respuesta)
    print()

PREGUNTA 1: ¿Qué es la tokenización?
--------------------------------------------------------------------------------
La **tokenización** es el proceso de convertir algo (como un activo o dato) en unidades digitales pequeñas y únicas, llamadas *tokens*.  

Por ejemplo:  
- En finanzas, se puede dividir una propiedad en tokens para que cualquiera pueda comprar una parte.  
- En criptomonedas, como Ethereum, los tokens representan activos o derechos digitales.  

Esto permite mayor acceso, transparencia y eficiencia en operaciones financieras o digitales.

PREGUNTA 2: ¿Cómo podría ayudar un chatbot a estudiantes de una biblioteca?
--------------------------------------------------------------------------------
Un chatbot puede ayudar a estudiantes de una biblioteca con:

- Respuestas rápidas a preguntas sobre libros, horarios o servicios.  
- Recomendaciones de libros según intereses (ej: ciencia, literatura, historia).  
- Guía para usar la biblioteca digital o bases de datos.  
- Ayuda

## 10. Versión para que cada alumno complete

Copiá esta celda y reemplazá las preguntas por las de tu propio chatbot.


In [ ]:
mis_preguntas = [
    "Escribí acá una pregunta conceptual sobre PLN",
    "Escribí acá una pregunta relacionada con tu futuro chatbot",
    "Escribí acá una pregunta con un dato específico que el modelo no debería inventar"
]

for numero, pregunta in enumerate(mis_preguntas, start=1):
    print("=" * 80)
    print(f"PREGUNTA {numero}: {pregunta}")
    print("-" * 80)
    respuesta = responder_con_ia(pregunta)
    print(respuesta)
    print()

PREGUNTA 1: Escribí acá una pregunta conceptual sobre PLN
--------------------------------------------------------------------------------
¡Claro! Por favor, escribe tu pregunta conceptual sobre Planteamiento de Lenguaje Natural (PLN). Estoy aquí para ayudarte. 😊

PREGUNTA 2: Escribí acá una pregunta relacionada con tu futuro chatbot
--------------------------------------------------------------------------------
¡Gracias por la pregunta! Aunque soy un chatbot, no tengo un "futuro" como ser humano. Sin embargo, estoy aquí para ayudarte hoy con cualquier duda o tema que necesites. ¿En qué puedo ayudarte? 😊

PREGUNTA 3: Escribí acá una pregunta con un dato específico que el modelo no debería inventar
--------------------------------------------------------------------------------
¡Claro! Por favor, escribe tu pregunta con el dato específico que deseas que se use. Así aseguro que no invente información y responda con precisión.



## 11. Tabla de análisis

Después de probar las tres preguntas, completá esta tabla en tu carpeta o en un documento.

| Pregunta | ¿Respondió bien? | ¿Fue claro? | ¿Inventó algo? | Observación |
|---|---|---|---|---|
| Pregunta conceptual | | | | |
| Pregunta aplicada al proyecto | | | | |
| Pregunta problemática | | | | |


# Entrega breve de la clase

Entregar:

1. Nombre tentativo del chatbot.
2. Tema.
3. Usuario objetivo.
4. Tres preguntas probadas con el modelo.
5. Una respuesta buena del modelo.
6. Una respuesta que habría que controlar o mejorar.

La respuesta problemática es útil porque nos muestra por qué más adelante vamos a necesitar:

- mejores prompts;
- límites;
- fallback;
- base de conocimiento;
- RAG.


# Cierre

Hoy logramos que el modelo genere respuestas.

Pero todavía no le dimos información propia ni documentos confiables.

En las próximas clases vamos a aprender a controlar mejor su comportamiento mediante prompts, memoria, reglas de PLN y finalmente RAG.
